### Loading the existing topology and pricing structure

In [22]:
import pricing_driven_service_allocation as pdsa
TOPOLOGIES_RESULT_DIR = "synthetic-dataset/synthetic-topologies"
DATASET_RESULT_DIR = "synthetic-dataset/data"
UNLIMITED_VALUE = 100000000
topology_id= "a75b90ab-051a-4160-a8d3-ba87aeb0a675"
C_LOCATIONS_DATASET_PATH = "eua-dataset/users/users-aus.csv"

### 1. Load all client locations

In [23]:
c_locations_df = pdsa.dataset.load_client_locations_dataframe(C_LOCATIONS_DATASET_PATH)

import os
c_locations_df.to_csv(os.path.join(DATASET_RESULT_DIR, "client_locations.csv"))

print("Dataset size:", len(c_locations_df))
c_locations_df.head()

Dataset size: 4748


,latitude,longitude
0,-30.5083,151.6712
1,-37.8833,145.3333
2,-32.2430,148.6048
3,-37.8141,144.9630
4,-27.4891,153.0188


### 2. Filter area and number of clients

In [24]:
import pandas as pd
from shapely.geometry import Point, Polygon

c_locations_df = pd.read_csv(os.path.join(DATASET_RESULT_DIR, "client_locations.csv"), index_col=0)

import yaml

# Load the YAML file
with open(os.path.join(DATASET_RESULT_DIR, "experiment_boundaries.yml"), 'r') as file:
    config_data = yaml.safe_load(file)

# Access experimental parameters
client_density = config_data['experiments']['small']['ar/vr']['demand']['clients_density']
client_area = config_data['experiments']['small']['ar/vr']['demand']['zone']

# Filter only those clients that are in the polygon
poly = Polygon(client_area)
mask = c_locations_df.apply(lambda row: poly.contains(Point(row['longitude'], row['latitude'])), axis=1)
f_area_df = c_locations_df[mask]

# Filter a subset of the clients to have density from [0,1]
f_clients_df = f_area_df.sample(n=int(len(f_area_df) * client_density), random_state=35)
f_clients_df.to_csv(os.path.join(DATASET_RESULT_DIR, "client_locations_filtered.csv"), index=False)

print("Dataset size:", len(f_clients_df))
f_clients_df.head()

Dataset size: 21


,latitude,longitude
3657,-37.8146,144.970
2813,-37.8137,144.974
3342,-37.8187,144.958
4177,-37.8168,144.966
2993,-37.8151,144.959


### 3. Transform clients into resource demand

In [25]:
from pricing_driven_service_allocation.generators.client_demand import calculate_resources, AppType

c_locations_df = pd.read_csv(os.path.join(DATASET_RESULT_DIR, "client_locations_filtered.csv"), index_col=0)


# --- Example Usage ---
n_clients = len(c_locations_df)
concurrency = config_data['experiments']['small']['ar/vr']['demand']['concurrency']
max_distance = config_data['experiments']['small']['ar/vr']['request']['max_distance']

request = calculate_resources(n_clients, AppType.AR_VR, concurrency)

print(f"Resources requested for AR/VR application", request)

Resources requested for AR/VR application {'service_mode': <AppType.AR_VR: 'ar/vr'>, 'active_users': 16, 'ram_total_gb': 129, 'gpu_total_tflops': 61, 'cpu_total_cores': 35}


### 4. Use demand for solving the pricing

In [26]:
import os
from iPricing.model.iPricing_pb2 import Pricing

# Convert YAML to Protocol Buffer using the package function
pricing_obj = pdsa.utils.yaml_to_pricing_proto(
    os.path.join(TOPOLOGIES_RESULT_DIR, topology_id, "pricing.yml"),
    Pricing
)

In [27]:
import json

# Custom request configuration
custom_request = {
    'currency': 'USD',
    'users_location': [tuple(coord) for coord in config_data['experiments']['small']['ar/vr']['demand']['zone']],
    # 'providers_to_consider': ["Telstra", "Optus", "Vodafone", "Telecom", "Macquarie"],
    'budget': config_data['experiments']['small']['ar/vr']['request']['budget'],
    'max_devices': 2,
    # 'device_types': ['CAMERA', 'SENSOR', 'DATA_CENTER'],
    'resources': {
        'available_ram': request['ram_total_gb'],
        # 'available_storage': 500,
        'available_vcpu': request['cpu_total_cores'],
        'available_gpu': request['gpu_total_tflops'],
        # 'available_tpu': 0,
    },
    'max_distance': config_data['experiments']['small']['ar/vr']['request']['max_distance'],  # in meters
}

# Generate problem instance using the package function
problem_instance_pricing, filter_criteria = pdsa.generators.problem_instance(
    instance_pricing=pricing_obj,
    request=custom_request,
    topologies_result_dir=TOPOLOGIES_RESULT_DIR,
    unlimited_value=UNLIMITED_VALUE
)

print("\nGenerated problem instance pricing and filter criteria:")
print(f"- Original number of add-ons: {len(pricing_obj.addOns)}")
print(f"- Number of add-ons: {len(problem_instance_pricing.addOns)}")
print(f"- Filter criteria: {json.dumps(filter_criteria, indent=2)}")


Generated problem instance pricing and filter criteria:
- Original number of add-ons: 1098
- Number of add-ons: 1098
- Filter criteria: {
  "maxPrice": 1000,
  "maxSubscriptionSize": 2,
  "usageLimits": {
    "available_ram": 129,
    "available_vcpu": 35,
    "available_gpu": 61,
    "distance": 99999975
  }
}


In [28]:
pdsa.utils.pricing_proto_to_yaml(
    problem_instance_pricing,
    os.path.join(TOPOLOGIES_RESULT_DIR, topology_id, "problem_instance_pricing.yml")
)

Generated pricing saved to: synthetic-dataset/synthetic-topologies/a75b90ab-051a-4160-a8d3-ba87aeb0a675/problem_instance_pricing.yml
